# Apparel Material Deduplication System
## Pipeline Step 1: Category Extraction & Raw Data Partitioning

**Author:** Senior Data Engineer  
**Objective:** Ingest raw apparel materials data, validate and isolate valid category records from missing/blank category entries, export category overview statistics with a missing records summary row to `data/overviews/categories_overview.csv`, and partition valid records into isolated category CSVs in `data/02_by_category/`.

### Pipeline Stages:
1. **Environment & Path Setup**: Dynamic project root detection and programmatic folder provisioning (`data/01_raw/`, `data/02_by_category/`, `data/overviews/`).
2. **Data Ingestion & Missing Category Detection**: Read raw data, identify and isolate valid vs. null/blank category records.
3. **Category Overview & Missing Summary Row Export**: Calculate statistics on valid categories and append a 'Missing Category Records' summary row at the bottom.
4. **Name Normalization**: Safe OS-compliant file naming helper.
5. **Valid Dataset Partitioning**: Partition only valid category records into `data/02_by_category/category_<normalized_slug>.csv`.
6. **Data Integrity & Reconciliation Audit**: Reconcile partitioned rows + missing rows against total raw ingested rows.

In [ ]:
import os
import re
from pathlib import Path
import pandas as pd

# -------------------------------------------------------------------------
# 1. Environment & Path Configuration
# -------------------------------------------------------------------------
CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ["notebooks", "notebook"] else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "01_raw"
PARTITION_DIR = DATA_DIR / "02_by_category"
OVERVIEWS_DIR = DATA_DIR / "overviews"
METADATA_FILE = OVERVIEWS_DIR / "categories_overview.csv"
UNCAT_FILE = OVERVIEWS_DIR / "category_uncategorized.csv"

# Ensure all directories exist programmatically
for directory in (RAW_DATA_DIR, PARTITION_DIR, OVERVIEWS_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print(f"[SETUP] Project Root:             {PROJECT_ROOT}")
print(f"[SETUP] Raw Data Dir:             {RAW_DATA_DIR}")
print(f"[SETUP] Partition Out Dir:        {PARTITION_DIR}")
print(f"[SETUP] Overviews Dir:            {OVERVIEWS_DIR}")
print(f"[SETUP] Categories Overview:      {METADATA_FILE}")
print(f"[SETUP] Category Uncategorized:   {UNCAT_FILE}")


In [ ]:
# -------------------------------------------------------------------------
# 2. Raw Data Ingestion & Missing Category Detection
# -------------------------------------------------------------------------
csv_candidates = list(RAW_DATA_DIR.glob("*.csv"))
if not csv_candidates:
    raise FileNotFoundError(f"No CSV dataset found in: {RAW_DATA_DIR}")

raw_csv_path = (
    RAW_DATA_DIR / "raw_materials.csv"
    if (RAW_DATA_DIR / "raw_materials.csv").exists()
    else csv_candidates[0]
)
df_raw = pd.read_csv(raw_csv_path)
print(f"[INGESTION] Loaded {len(df_raw)} records from {raw_csv_path.name}")

# Standardize column naming
col_map = {col: col.strip() for col in df_raw.columns}
df_raw = df_raw.rename(columns=col_map)

category_col = next((col for col in df_raw.columns if col.lower() == "category"), None)
if category_col is None:
    raise KeyError(f"Mandatory column 'Category' not found. Available: {list(df_raw.columns)}")

if category_col != "Category":
    df_raw = df_raw.rename(columns={category_col: "Category"})

# Detect rows where Category is null, NaN, or whitespace
is_missing_category = (
    df_raw["Category"].isna() |
    (df_raw["Category"].astype(str).str.strip() == "") |
    (df_raw["Category"].astype(str).str.strip().str.lower().isin(["nan", "none", "null"]))
)

df_valid = df_raw[~is_missing_category].copy()
df_missing = df_raw[is_missing_category].copy()

# Standardize whitespace on valid category values
df_valid["Category"] = df_valid["Category"].astype(str).str.strip()

print(f"[VALIDATION] Total raw records:            {len(df_raw)}")
print(f"[VALIDATION] Valid category records:       {len(df_valid)}")
print(f"[VALIDATION] Missing/blank category records: {len(df_missing)}")

# Display sample valid records
df_valid.head()

In [ ]:
# -------------------------------------------------------------------------
# 3. Category Overview & Category Uncategorized Separate Export
# -------------------------------------------------------------------------
total_raw_records = len(df_raw)

# Separate named categories from uncategorized/missing categories
is_explicit_uncat = df_valid["Category"].astype(str).str.strip().str.lower().isin(["uncategorized", "category_uncategorized"])
df_named = df_valid[~is_explicit_uncat].copy()
df_uncat_all = pd.concat([df_valid[is_explicit_uncat], df_missing], ignore_index=True)

# 1. Summary statistics for named categories ONLY (sorted descending by record count)
category_stats = df_named.groupby("Category", as_index=False).agg(
    record_count=("Category", "count"),
    unique_material_names=("Material name", "nunique") if "Material name" in df_named.columns else ("Category", "count")
)
category_stats["percentage_of_total"] = (category_stats["record_count"] / total_raw_records * 100).round(2)
category_stats = category_stats.sort_values(by=["record_count", "Category"], ascending=[False, True]).reset_index(drop=True)

# Export categories_overview.csv (valid named categories ONLY)
OVERVIEWS_DIR.mkdir(parents=True, exist_ok=True)
category_stats.to_csv(METADATA_FILE, index=False)
print(f"[METADATA] Exported valid categories overview to: {METADATA_FILE}")

# 2. Summary statistics for Uncategorized records as a SEPARATE file: category_uncategorized.csv
missing_count = len(df_uncat_all)
missing_unique_materials = (
    df_uncat_all["Material name"].nunique()
    if ("Material name" in df_uncat_all.columns and missing_count > 0)
    else 0
)
missing_percentage = round(missing_count / total_raw_records * 100, 2) if total_raw_records > 0 else 0.0

uncat_summary_df = pd.DataFrame([{
    "Category": "category_uncategorized",
    "record_count": missing_count,
    "unique_material_names": missing_unique_materials,
    "percentage_of_total": missing_percentage
}])

# Export to D:\Projects\POC_Dedup\data\overviews\category_uncategorized.csv
uncat_summary_df.to_csv(UNCAT_FILE, index=False)
print(f"[METADATA] Exported category_uncategorized count separately to: {UNCAT_FILE}")

print("\n--- Valid Categories Overview (categories_overview.csv) ---")
print(category_stats.to_string(index=False))

print("\n--- Category Uncategorized Count (Separate category_uncategorized.csv) ---")
print(uncat_summary_df.to_string(index=False))


In [ ]:
# -------------------------------------------------------------------------
# 4. Name Normalization Helper
# -------------------------------------------------------------------------
def normalize_category_name(category_name: str) -> str:
    """
    Normalizes category names into safe, OS-compliant filename slugs.
    Example: 'Packaging / Boxing' -> 'packaging_boxing'
    """
    if not category_name or pd.isna(category_name):
        return "uncategorized"
    clean = re.sub(r"[^a-zA-Z0-9]+", "_", str(category_name).strip().lower())
    return clean.strip("_") or "uncategorized"

# Unit tests on normalization helper
test_cases = ["Fabric", "Packaging / Boxing", " Trim & Accessories ", "Thread/Yarn"]
for tc in test_cases:
    print(f"  {repr(tc):<25} -> {repr(normalize_category_name(tc))}")

In [ ]:
# -------------------------------------------------------------------------
# 5. Valid Dataset Partitioning & Export
# -------------------------------------------------------------------------
# Assign normalized slug only to valid records
df_uncat_part = df_uncat_all.copy()
df_uncat_part["Category"] = "Uncategorized"
df_to_partition = pd.concat([df_named, df_uncat_part], ignore_index=True)
df_to_partition["norm_category"] = df_to_partition["Category"].apply(normalize_category_name)

generated_partitions = {}
print("[PARTITIONING] Writing valid category partitions to disk:")

for norm_cat, group in df_to_partition.groupby("norm_category"):
    filename = f"category_{norm_cat}.csv"
    output_path = PARTITION_DIR / filename
    
    # Drop internal helper column before saving
    export_df = group.drop(columns=["norm_category"])
    export_df.to_csv(output_path, index=False)
    
    generated_partitions[norm_cat] = {
        "filename": filename,
        "path": str(output_path),
        "records": len(export_df)
    }
    print(f"  [OK] {filename:<32} | {len(export_df):>3} records | Path: {output_path.name}")

print(f"\n[PARTITIONING] Successfully generated {len(generated_partitions)} partition files in {PARTITION_DIR}")

In [ ]:
# -------------------------------------------------------------------------
# 6. Data Integrity & Reconciliation Audit
# -------------------------------------------------------------------------
total_partitioned = sum(p["records"] for p in generated_partitions.values())

# Verify full mathematical reconciliation: Valid Partitioned + Missing == Total Ingested
assert total_partitioned == total_raw_records, (
    f"Reconciliation error! Partitioned ({total_partitioned}) != Total Raw ({total_raw_records})"
)

print("=" * 60)
print("PIPELINE STEP 1 EXECUTION SUMMARY")
print("=" * 60)
print(f"Total Raw Records Ingested:          {total_raw_records}")
print(f"Valid Records Partitioned:           {total_partitioned}")
print(f"Missing Category Records:            {missing_count}")
print(f"Total Partition Files Created:       {len(generated_partitions)}")
print(f"Overview Metadata Exported:          {METADATA_FILE}")
print(f"Data Reconciliation Check:           100% matched (0 uncounted records)")
print("=" * 60)
print(f"[SUCCESS] Valid records cleanly partitioned with complete metadata overview and missing summary.")

## 7. Visual Summary  <!-- VIS_CELL -->
Category distribution and partition reconciliation.

In [ ]:
# VIS_CELL
try:
    from IPython.display import display
except ImportError:
    display = print
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True})
PALETTE = ["#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD", "#937860"]

overview = pd.concat([category_stats, uncat_summary_df.assign(Category="Uncategorized")], ignore_index=True)
overview = overview.sort_values("record_count", ascending=False).reset_index(drop=True)
display(overview)

fig, ax = plt.subplots(1, 3, figsize=(18, 4.8))
bars = ax[0].bar(overview["Category"], overview["record_count"], color=PALETTE[:len(overview)])
ax[0].bar_label(bars, fmt="{:,.0f}", padding=2)
ax[0].set_title("Records per category"); ax[0].tick_params(axis="x", rotation=30)

ax[1].pie(overview["record_count"], labels=overview["Category"], autopct="%1.1f%%",
          colors=PALETTE[:len(overview)], startangle=90, wedgeprops={"width": 0.45})
ax[1].set_title("Share of total records")

bars = ax[2].bar(overview["Category"], overview["unique_material_names"], color=PALETTE[:len(overview)])
ax[2].bar_label(bars, padding=2)
ax[2].set_title("Distinct raw material names per category"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

recon = pd.DataFrame({
    "Metric": ["Raw records ingested", "Valid-category records", "Missing/blank category", "Records written to partitions"],
    "Count": [total_raw_records, len(df_valid), len(df_missing), total_partitioned],
})
display(recon)
